# J11 — Phương pháp hoàn chỉnh lấy cảm hứng từ cách con người hiểu video (bộ câu pilot)

**Tài khoản Kaggle #13.** GPU **T4 ×2**, bật **Internet** (Session options), rồi **Save Version → Save & Run All**. Thời gian ≈ 3,5–4 giờ. Khi xong, tải **`j11_full_pilot.zip`** ở cell cuối và gửi cho Claude.

Notebook tự chạy thử 4 mẫu trước; nếu có lỗi thật (không tính clip quá dài), nó dừng ngay để không phí GPU. Job tự dừng an toàn trước hạn 12 giờ của Kaggle và giữ những gì đã chạy xong.

### Vì sao chạy
Ghép các bước đã được chứng minh riêng lẻ: (1) ĐỊNH HƯỚNG — mô hình tự tìm lúc có âm thanh/sự kiện được hỏi qua attention lên token audio, khử độ nổi bật bằng PMI (J8/J9: đúng đoạn 97–100%); (2) NÓI THẦM — mô hình tự diễn đạt điều nó nghe thành một cụm từ (trí nhớ làm việc bằng lời; debug: độ chênh giữa các vùng tăng từ ~0,5 lên 2–18 logit); (3) NHÌN KHI TẮT TIẾNG — với mỗi vùng trước/trong/sau mốc, hỏi "có thấy … không" khi câu hỏi bị tắt audio và chỉ nhìn vùng đó ở lớp 13–27 (định tuyến attention, ngang cắt clip; debug 5/5); (4) TRÍ NHỚ SỰ KIỆN CÓ NGUỒN — dòng thời gian tách nội dung lời nói khỏi điều nhìn thấy (sửa lỗi nhầm nguồn); (5) SUY NGHĨ CÓ CHỦ Ý — suy luận từng bước, đọc đáp án từ lời suy luận (TF: một bước 27–46%, từng bước 86%).

### Đo gì
Mỗi câu: base; acar (một lượt, mốc tự động + tắt audio, tốt nhất của J8); plain (pipeline, trả lời một bước); cot (pipeline, suy luận từng bước, 3 thứ tự lựa chọn bỏ phiếu). Lưu mọi bước trung gian (mốc, câu nói thầm, điểm nhìn từng vùng, dòng thời gian, lời suy luận) để phân tích từng câu sai.

### Đọc kết quả thế nào (tiêu chí định trước)
cot ≥ 90% ở cả TA, TF, CMTB → phương pháp đạt mục tiêu trên pilot, chuyển sang held-out một lần. Nếu không: phân tích từng câu sai theo bước (mốc sai / nói thầm sai / nhìn sai / suy luận sai) để sửa đúng bước.

Không có nhãn nào được đưa vào mô hình; nhãn chỉ dùng để chấm điểm.

In [ ]:
# CELL 0 — kiểm tra Internet
import socket
for host in ('pypi.org', 'huggingface.co'):
    try:
        socket.create_connection((host, 443), timeout=8).close(); print('OK  ', host)
    except OSError as e:
        raise RuntimeError('Không kết nối được ' + host + '. Bật Internet: Session options -> Internet = On.') from e


In [ ]:
# CELL 1 — cài môi trường. Chạy một lần cho mỗi session Kaggle.
import os, sys, json, subprocess, importlib.metadata, venv, time, html, hashlib, zipfile, shutil
from pathlib import Path
from IPython.display import display, Video, Markdown, HTML, FileLink
assert Path('/kaggle').exists(), 'Dùng Kaggle Edit với GPU T4 x2 và Internet.'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTHONIOENCODING']='utf-8'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
os.environ['FORCE_QWENVL_VIDEO_READER']='decord'  # torchvision.io.read_video was removed in Kaggle's build.
ROOT=Path('/kaggle/working/svhalluc_prompt_lab_runner');ROOT.mkdir(parents=True,exist_ok=True)
if 'OUT' not in globals():
    OUT=Path('/kaggle/working/svhalluc_prompt_lab_'+time.strftime('%Y%m%d_%H%M%S'))
    OUT.mkdir(parents=True,exist_ok=False)
SOURCES={'lab_client.py': '"""Standard-library client; the notebook kernel never imports/reloads the model."""\nimport json\nimport queue\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\nfrom uuid import uuid4\n\nPREFIX=\'__SVHLAB_JSON__\'\n\n\nclass PromptLab:\n    def __init__(self, python, root, output, mock=False):\n        self.root=Path(root);self.output=Path(output);self.output.mkdir(parents=True,exist_ok=True)\n        self.log_path=self.output/\'worker.log\'\n        self.log=self.log_path.open(\'a\',encoding=\'utf-8\')\n        self.responses=queue.Queue()\n        command=[str(python),\'-u\',str(self.root/\'lab_worker.py\'),\'--output\',str(self.output)]\n        if mock:command.append(\'--mock\')\n        self.process=subprocess.Popen(command,cwd=str(self.root),stdin=subprocess.PIPE,stdout=subprocess.PIPE,\n            stderr=self.log,text=True,encoding=\'utf-8\',errors=\'replace\',bufsize=1)\n        self.reader=threading.Thread(target=self._read,daemon=True);self.reader.start()\n        self.ready=self._wait(None,3600)\n        if self.ready.get(\'event\')!=\'ready\':raise RuntimeError(self.ready)\n\n    def _read(self):\n        for line in self.process.stdout:\n            if line.startswith(PREFIX):\n                try:self.responses.put(json.loads(line[len(PREFIX):]))\n                except Exception:self.responses.put({\'event\':\'fatal\',\'error\':\'Malformed worker response\'})\n\n    def _wait(self, request_id, timeout):\n        started=time.monotonic();last_print=started\n        while time.monotonic()-started<timeout:\n            try:response=self.responses.get(timeout=1)\n            except queue.Empty:\n                if self.process.poll() is not None:raise RuntimeError(\'Worker stopped; see \'+str(self.log_path))\n                if time.monotonic()-last_print>=30:\n                    print(\'Đang tải/chạy model; log:\',self.log_path,flush=True);last_print=time.monotonic()\n                continue\n            if response.get(\'event\')==\'fatal\':raise RuntimeError(response.get(\'error\'))\n            if request_id is None or response.get(\'request_id\')==request_id:return response\n            # Late reply after a notebook interrupt is not reused for a newer prompt.\n        raise TimeoutError(\'Worker timed out; see \'+str(self.log_path))\n\n    def is_alive(self):return self.process.poll() is None\n\n    def request(self, payload, timeout=1800):\n        if not self.is_alive():raise RuntimeError(\'Run the model-loading cell again\')\n        request_id=uuid4().hex\n        self.process.stdin.write(json.dumps({**payload,\'request_id\':request_id},ensure_ascii=False)+\'\\n\')\n        self.process.stdin.flush()\n        response=self._wait(request_id,timeout)\n        if not response.get(\'ok\'):raise RuntimeError(response.get(\'error\'))\n        return response[\'result\']\n\n    def close(self):\n        if self.is_alive():self.process.terminate();self.process.wait(timeout=30)\n        self.log.close()\n', 'lab_native_video.py': '"""Native video-file input through Qwen\'s official utilities; no evidence extraction."""\nfrom pathlib import Path\n\nSYSTEM=\'You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech.\'\n# T4 SDPA cannot hold the default ~18k-token attention matrix. Keep Qwen\'s\n# temporal sampling and original MP4 path, while bounding only pixels per frame.\nVIDEO_MAX_PIXELS=160*28*28\n\n\ndef conversation_for(path, text, media_mode=\'av\', max_pixels=VIDEO_MAX_PIXELS):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    media=({\'type\':\'audio\',\'audio\':str(path)} if media_mode==\'audio\' else\n           {\'type\':\'video\',\'video\':str(path),\'max_pixels\':max_pixels})\n    return [\n        {\'role\':\'system\',\'content\':[{\'type\':\'text\',\'text\':SYSTEM}]},\n        {\'role\':\'user\',\'content\':[media,\n                                  {\'type\':\'text\',\'text\':text}]},\n    ]\n\n\ndef prepare_video(path, max_pixels=VIDEO_MAX_PIXELS):\n    from qwen_omni_utils import process_mm_info\n    path=Path(path).resolve()\n    if not path.is_file():raise FileNotFoundError(path)\n    # MP4 path, not caller-extracted frames. Keep Qwen\'s default temporal sampling;\n    # max_pixels per frame only bounds resolution for the T4 memory budget.\n    # The notebook selects Qwen\'s supported Decord reader because Kaggle\'s\n    # torchvision build no longer provides torchvision.io.read_video.\n    audios,images,videos,video_kwargs=process_mm_info(\n        conversation_for(path,\'\',max_pixels=max_pixels),use_audio_in_video=True,return_video_kwargs=True)\n    if images is not None:raise RuntimeError(\'Unexpected supplemental images\')\n    if audios is None or len(audios)!=1 or videos is None or len(videos)!=1:\n        raise RuntimeError(\'Expected one original video and its audio\')\n    # Transformers 4.57.6 Omni expects a scalar fps for this single video.\n    fps=video_kwargs[\'fps\']\n    if isinstance(fps,(list,tuple)):\n        if len(fps)!=1:raise RuntimeError(\'Expected one video fps\')\n        fps=fps[0]\n    return {\'path\':str(path),\'audio\':audios,\'videos\':videos,\'fps\':float(fps),\n            \'max_pixels_per_frame\':max_pixels}\n\n\ndef qwen_inputs(processor, prompt, media, media_mode=\'av\'):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    use_audio=media_mode in (\'av\',\'audio\')\n    use_video=media_mode in (\'av\',\'vision\')\n    rendered=processor.apply_chat_template(conversation_for(media[\'path\'],prompt,media_mode,media[\'max_pixels_per_frame\']),\n        add_generation_prompt=True,tokenize=False)\n    inputs=processor(text=rendered,audio=media[\'audio\'] if use_audio else None,images=None,videos=media[\'videos\'] if use_video else None,\n        return_tensors=\'pt\',padding=True,\n        videos_kwargs={\'fps\':media[\'fps\'],\'use_audio_in_video\':media_mode==\'av\'})\n    if (\'input_features\' in inputs)!=use_audio or (\'pixel_values_videos\' in inputs)!=use_video:\n        raise RuntimeError(\'Native inputs do not match the requested modalities\')\n    if \'pixel_values\' in inputs:raise RuntimeError(\'Unexpected supplemental image tokens\')\n    # HF Transformers takes this switch from videos_kwargs, not its top-level\n    # kwargs. Check that the resulting token sequence actually contains audio.\n    audio_token_id=processor.tokenizer.convert_tokens_to_ids(processor.audio_token)\n    audio_tokens=int((inputs[\'input_ids\']==audio_token_id).sum().item())\n    expected_audio_tokens=0\n    if use_audio:\n        if \'feature_attention_mask\' not in inputs:\n            raise RuntimeError(\'Audio feature mask missing; cannot verify audio-token alignment\')\n        input_lengths=(inputs[\'feature_attention_mask\'].sum(-1)-1)//2+1\n        expected_audio_tokens=int((((input_lengths-2)//2+1).clamp_min(0)).sum().item())\n    if audio_tokens!=expected_audio_tokens or (use_audio and audio_tokens==0):\n        raise RuntimeError(f\'Audio tokens missing/misaligned: found {audio_tokens}, expected {expected_audio_tokens}\')\n    expected_video_tokens=0\n    if use_video:\n        grid=inputs[\'video_grid_thw\']\n        merge=processor.video_processor.merge_size**2\n        expected_video_tokens=int((grid.prod(-1).sum()//merge).item())\n    video_token_id=processor.tokenizer.convert_tokens_to_ids(processor.video_token)\n    video_tokens=int((inputs[\'input_ids\']==video_token_id).sum().item())\n    if video_tokens!=expected_video_tokens or (use_video and video_tokens==0):\n        raise RuntimeError(f\'Video tokens missing/misaligned: found {video_tokens}, expected {expected_video_tokens}\')\n    return inputs,rendered\n\n\ndef move_inputs(inputs, model):\n    import torch\n    inputs=inputs.to(model.get_input_embeddings().weight.device)\n    for key in (\'input_features\',\'pixel_values_videos\'):\n        if key in inputs:inputs[key]=inputs[key].to(torch.float16)\n    return inputs\n', 'lab_setup.py': '"""Create/repair a venv without ensurepip; use the host pip to manage it."""\nimport importlib.metadata\nimport json\nimport subprocess\nimport sys\nimport venv\nfrom pathlib import Path\n\n\ndef prepare_environment(directory):\n    directory=Path(directory).resolve()\n    version=importlib.metadata.version(\'pip\')\n    parts=version.split(\'.\')\n    if tuple(int(part) for part in parts[:2]) < (22,3):\n        raise RuntimeError(\'Host pip >=22.3 is required for --python; found \'+version)\n    # Repeat even when the interpreter exists: an earlier ensurepip failure leaves\n    # an incomplete venv. No deletion, pip bootstrap or system package changes.\n    venv.EnvBuilder(system_site_packages=True,with_pip=False).create(directory)\n    python=directory/(\'Scripts/python.exe\' if sys.platform==\'win32\' else \'bin/python\')\n    completed=subprocess.run([str(python),\'-c\',\n        \'import json,sys; print(json.dumps({"prefix":sys.prefix,"base":sys.base_prefix}))\'],\n        check=True,capture_output=True,text=True,encoding=\'utf-8\')\n    info=json.loads(completed.stdout)\n    if Path(info[\'prefix\']).resolve()!=directory or info[\'prefix\']==info[\'base\']:\n        raise RuntimeError(\'The target interpreter is not the requested virtual environment\')\n    # Official pip --python also supports targets without their own pip module.\n    return python,[sys.executable,\'-m\',\'pip\',\'--python\',str(python)]\n', 'lab_worker.py': '"""Persistent, frozen Qwen Omni worker: one model load, fresh context for each prompt."""\nfrom __future__ import annotations\nimport argparse\nimport gc\nimport json\nimport re\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\nfrom uuid import uuid4\n\nWIRE = sys.stdout\nPREFIX = "__SVHLAB_JSON__"\nMODEL = "Qwen/Qwen2.5-Omni-3B"\nREVISION = "f75b40e3da2003cdd6e1829b1f420ca70797c34e"\n\n\ndef emit(value):\n    WIRE.write(PREFIX + json.dumps(value, ensure_ascii=False) + "\\n"); WIRE.flush()\n\n\ndef parse_final(text, complete):\n    """Read the explicit final choice, with support for older prompt formats."""\n    if not complete: return None\n    final_labels = re.findall(r"(?i)\\bFinal answer\\s*:", text)\n    if len(final_labels)>1:return None\n    final_match = re.search(r"(?i)\\bFinal answer\\s*:\\s*([AB](?:\\s*\\((?:Yes|No)\\))?)\\s*[.!?]?\\s*$",text)\n    if final_labels and not final_match:return None\n    final_fields=[final_match.group(1)] if final_match else []\n    fields = re.findall(r"(?im)^\\s*4\\.\\s*Original question\\s*:\\s*([^\\n]+)", text)\n    if len(fields)>1:return None\n    terminal = re.search(r"(?i)\\b(?:the\\s+)?answer\\s+is\\s+([AB](?:\\s*\\((?:Yes|No)\\))?)[.!]?\\s*$",text)\n    value=(final_fields[0] if final_fields else fields[0] if fields else terminal.group(1) if terminal else text).strip()\n    match=re.match(r"^(?:Answer\\s*:\\s*)?([AB])(?=$|[\\s().:—-])",value,re.I)\n    if not match or re.search(r"\\bor\\s+[AB](?:\\b|\\s*\\()",value,re.I):return None\n    letter=match.group(1).upper()\n    meaning=re.match(r"\\s*\\((Yes|No)\\)",value[match.end():],re.I)\n    if meaning and (letter,meaning.group(1).lower()) not in (("A","yes"),("B","no")):return None\n    return letter\n\n\ndef parse_baseline(text,complete):\n    if not complete:return None\n    value=text.strip()\n    match=re.match(r"^(?:(?:The\\s+)?(?:answer|option)\\s*(?:is|:)\\s*)?([AB])\\b",value,re.I)\n    if match:\n        if re.search(r"\\bor\\s+[AB]\\b",value,re.I):return None\n        letter=match.group(1).upper()\n        meaning=re.match(r"\\s*\\((Yes|No)\\)",value[match.end():],re.I)\n        if meaning and (letter,meaning.group(1).lower()) not in ((\'A\',\'yes\'),(\'B\',\'no\')):return None\n        return letter\n    match=re.match(r"^(Yes|No)\\b",value,re.I)\n    return (\'A\' if match.group(1).lower()==\'yes\' else \'B\') if match else None\n\n\ndef final_format_issues(text, complete):\n    issues = []\n    if not complete: issues.append("Output hit the token limit: final answer is not counted.")\n    if complete and parse_final(text, complete) is None:\n        issues.append("Could not identify one unambiguous Final answer: A (Yes) or B (No), or a supported older answer format.")\n    return issues\n\n\nclass Engine:\n    def __init__(self, output):\n        import torch\n        from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration\n        from transformers.models.qwen2_5_omni.configuration_qwen2_5_omni import Qwen2_5OmniThinkerConfig\n        self.output = Path(output); self.output.mkdir(parents=True, exist_ok=True)\n        torch.set_num_threads(2); torch.manual_seed(20261009)\n        if torch.cuda.device_count() < 2: raise RuntimeError("Enable Kaggle T4 x2")\n        config = Qwen2_5OmniThinkerConfig.from_pretrained(MODEL, revision=REVISION)\n        self.model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(\n            MODEL, revision=REVISION, config=config, torch_dtype=torch.float16,\n            attn_implementation="sdpa", device_map="balanced",\n            max_memory={0:"10GiB",1:"10GiB"}, low_cpu_mem_usage=True).eval()\n        for p in self.model.parameters(): p.requires_grad_(False)\n        if any(str(d) in ("cpu","disk") for d in self.model.hf_device_map.values()):\n            raise RuntimeError("Unexpected CPU offload; check both GPUs are available")\n        self.processor = Qwen2_5OmniProcessor.from_pretrained(MODEL, revision=REVISION)\n        self.meta = {"model":MODEL,"revision":REVISION,"device_map":self.model.hf_device_map,\n                     "runtime_parameters":sum(p.numel() for p in self.model.parameters()),\n                     "diagnostic_support":1}\n        self.media = self.row = None\n        (self.output / "model.json").write_text(json.dumps(self.meta, indent=2), encoding="utf-8")\n\n    def sample(self, request):\n        from lab_native_video import prepare_video\n        allowed = {"cmd","request_id","item_id","question","path"}\n        if set(request)-allowed: raise ValueError("Sample request has unexpected fields, possibly a label")\n        self.media = None; self.row = None\n        media = prepare_video(request[\'path\'])\n        self.media = media\n        self.row = {k:request[k] for k in ("item_id","question","path")}\n        result = {**self.row,\'frame_count\':int(media[\'videos\'][0].shape[0]),\n                  \'video_shape\':list(media[\'videos\'][0].shape),\'sampled_fps\':media[\'fps\'],\n                  \'audio_duration_s\':len(media[\'audio\'][0])/16000,\n                  \'preprocessing\':\'Official qwen-omni-utils 0.0.8, direct MP4 path, default temporal sampling, bounded max_pixels per frame for T4\',\n                  \'max_pixels_per_frame\':media[\'max_pixels_per_frame\'],\n                  \'same_media_for_both_conditions\':True}\n        (self.output/(request["item_id"]+"_media.json")).write_text(json.dumps(result,indent=2),encoding="utf-8")\n        return result\n\n    def generate(self, prompt, limit, baseline=False, media_mode=\'av\', diagnostic=False):\n        import torch\n        from lab_native_video import qwen_inputs, move_inputs\n        if self.media is None: raise RuntimeError("Select/load one sample before inference")\n        if not isinstance(limit,int) or not 16 <= limit <= 1024: raise ValueError("Token limit must be 16..1024")\n        inputs = None\n        started = time.perf_counter()\n        try:\n            inputs, rendered = qwen_inputs(self.processor, prompt, self.media,media_mode)\n            capacity=self.model.config.text_config.max_position_embeddings\n            if inputs[\'input_ids\'].shape[-1]+limit > capacity:\n                raise RuntimeError(f\'Native video + prompt exceed the model context ({capacity}); no media was silently truncated\')\n            inputs = move_inputs(inputs,self.model)\n            for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)\n            eos = {self.processor.tokenizer.eos_token_id}\n            original_eos = self.model.generation_config.eos_token_id\n            if isinstance(original_eos,int): eos.add(original_eos)\n            elif original_eos: eos.update(original_eos)\n            eos.discard(None)\n            # No previous chat or past_key_values is supplied; only the model/media are reused.\n            with torch.inference_mode():\n                output = self.model.generate(**inputs, use_audio_in_video=media_mode==\'av\', do_sample=False,\n                    num_beams=1, use_cache=True, max_new_tokens=limit, eos_token_id=sorted(eos),\n                    pad_token_id=self.processor.tokenizer.pad_token_id,\n                    return_dict_in_generate=True, output_logits=False)\n            ids = output.sequences[0,inputs[\'input_ids\'].shape[-1]:]\n            complete = bool(len(ids) and int(ids[-1]) in eos)\n            raw = self.processor.tokenizer.decode(ids,skip_special_tokens=True).strip()\n            result = {"item_id":self.row[\'item_id\'],"question":self.row[\'question\'],"prompt":prompt,\n                "rendered_prompt":rendered,"raw":raw,"complete":complete,"generated_tokens":len(ids),\n                "max_new_tokens":limit,"input_tokens":int(inputs[\'input_ids\'].shape[-1]),\n                "prediction":None if diagnostic else parse_baseline(raw,complete) if baseline else parse_final(raw,complete),\n                "format_issues":final_format_issues(raw,complete) if not (baseline or diagnostic) else [],\n                "latency_s":time.perf_counter()-started,\n                "peak_allocated_GiB":[round(torch.cuda.max_memory_allocated(i)/2**30,3) for i in range(torch.cuda.device_count())],\n                "method":"diagnostic" if diagnostic else "baseline" if baseline else "prompt","fresh_context":True,\n                "media_mode":media_mode,"audio_provided":media_mode in (\'av\',\'audio\'),\n                "video_provided":media_mode in (\'av\',\'vision\'),\n                "max_pixels_per_frame":self.media[\'max_pixels_per_frame\'],\n                "video_shape":list(self.media[\'videos\'][0].shape) if media_mode!=\'audio\' else None,\n                "run_id":uuid4().hex[:12]}\n            result[\'decision_interface\']=\'Read the freely generated answer; no logit adjustment or forced A/B selection\'\n            result[\'media_preprocessing\']=\'official_direct_video_defaults\'\n            path=self.output/(self.row[\'item_id\']+\'_\'+result[\'method\']+\'_\'+result[\'run_id\']+\'.json\')\n            path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n            result[\'result_path\']=str(path)\n            return result\n        finally:\n            inputs=None; gc.collect(); torch.cuda.empty_cache()\n\n    def dispatch(self, request):\n        command=request[\'cmd\']\n        if command==\'sample\':return self.sample(request)\n        if command==\'baseline\':\n            return self.generate(self.row[\'question\'],request.get(\'max_new_tokens\',384),True)\n        if command==\'prompt\':return self.generate(request[\'prompt\'],request.get(\'max_new_tokens\',384))\n        if command==\'diagnostic\':\n            if set(request)-{\'cmd\',\'request_id\',\'prompt\',\'media_mode\',\'max_new_tokens\',\'max_pixels\'}:\n                raise ValueError(\'Unexpected diagnostic fields; no labels are accepted\')\n            mode=request.get(\'media_mode\',\'av\')\n            if mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n            if self.media is None:raise RuntimeError(\'Select one sample first\')\n            current=self.media\n            pixels=request.get(\'max_pixels\',current[\'max_pixels_per_frame\'])\n            if pixels not in (current[\'max_pixels_per_frame\'],2*current[\'max_pixels_per_frame\']):\n                raise ValueError(\'Diagnostic resolution must be unchanged or doubled\')\n            try:\n                if pixels!=current[\'max_pixels_per_frame\']:\n                    from lab_native_video import prepare_video\n                    self.media=prepare_video(self.row[\'path\'],pixels)\n                    if self.media[\'fps\']!=current[\'fps\'] or self.media[\'videos\'][0].shape[0]!=current[\'videos\'][0].shape[0]:\n                        raise RuntimeError(\'Resolution probe changed temporal sampling\')\n                return self.generate(request[\'prompt\'],request.get(\'max_new_tokens\',192),media_mode=mode,diagnostic=True)\n            finally:self.media=current\n        if command==\'ping\':return self.meta\n        raise ValueError(\'Unknown command: \'+command)\n\n\ndef main():\n    sys.stdout = sys.stderr  # Library prints/progress cannot corrupt the JSON protocol.\n    parser=argparse.ArgumentParser();parser.add_argument(\'--output\',required=True);parser.add_argument(\'--mock\',action=\'store_true\')\n    args=parser.parse_args()\n    try:\n        engine=Engine(args.output) if not args.mock else None\n        emit({\'event\':\'ready\',\'model\':MODEL,\'revision\':REVISION,\'mock\':args.mock})\n    except Exception:\n        emit({\'event\':\'fatal\',\'error\':traceback.format_exc()});return\n    for line in sys.stdin:\n        request_id=None\n        try:\n            request=json.loads(line);request_id=request.get(\'request_id\')\n            if request[\'cmd\']==\'close\':emit({\'request_id\':request_id,\'ok\':True,\'result\':{}});break\n            result=engine.dispatch(request) if engine else {\'mock\':True,\'echo\':request}\n            emit({\'request_id\':request_id,\'ok\':True,\'result\':result})\n        except Exception:\n            emit({\'request_id\':request_id,\'ok\':False,\'error\':traceback.format_exc()})\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in SOURCES.items():(ROOT/name).write_text(source,encoding='utf-8')
HISTORICAL={'item_id': 'svh_0630', 'question': 'Is the thinly sliced scallions visible in the video? (A) Yes (B) No', 'prediction': 'A', 'gold': 'B', 'prompt': 'Is the thinly sliced scallions visible in the video? (A) Yes (B) No\nAnswer with exactly one option letter, A or B. Do not write an explanation.', 'method': 'AV_baseline', 'correct': 'False', 'raw': 'A', 'condition': 'question_only_historical_AV_baseline', 'source_zip': 'results.zip', 'source_member': 'svhalluc_external_evidence_20261008_070149/predictions.csv', 'note': 'Historical question-only AV_baseline log: prediction A, gold B. The notebook freshly reruns the baseline and the edited prompt on this sample; outputs can vary with runtime/model preprocessing.'}
(OUT/'historical_default.json').write_text(json.dumps(HISTORICAL,ensure_ascii=False,indent=2),encoding='utf-8')
if 'LAB' in globals() and LAB.is_alive():
    raise RuntimeError('Model đang chạy. Chỉ chạy cell PROMPT; nếu cần cài lại, gọi LAB.close() trước.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from lab_setup import prepare_environment
ENV=ROOT/'env'
PYTHON,PIP_COMMAND=prepare_environment(ENV)
print('Venv sẵn sàng (không dùng ensurepip):',PYTHON,flush=True)
constraints=[]
for package in ('torch','torchvision','torchaudio'):
    try:constraints.append(package+'=='+importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:pass
(ROOT/'keep_torch.txt').write_text('\n'.join(constraints)+'\n',encoding='utf-8')
packages=['transformers==4.57.6','accelerate==1.12.0','huggingface-hub==0.36.0','qwen-omni-utils==0.0.8','decord==0.6.0',
          'av==16.0.1','librosa==0.11.0','soundfile>=0.12,<0.14','numpy>=2,<2.4',
          'Pillow>=10,<13','sentencepiece>=0.2,<0.3','protobuf>=5,<7']
command=[*PIP_COMMAND,'install','--only-binary=:all:',
         '--constraint',str(ROOT/'keep_torch.txt'),*packages]
with (OUT/'install.log').open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,encoding='utf-8',errors='replace')
    for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
    code=process.wait()
if code:raise RuntimeError('Package installation failed; see '+str(OUT/'install.log'))
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=True)
subprocess.run([str(PYTHON),'-c','import torch, av, decord; from qwen_omni_utils.v2_5.vision_process import get_video_reader_backend; from transformers import Qwen2_5OmniThinkerForConditionalGeneration; assert torch.cuda.device_count()>=1; assert get_video_reader_backend()=="decord"; print("torch",torch.__version__,"decord",decord.__version__)'],check=True)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from lab_client import PromptLab
print('Setup ready. Kết quả và log:',OUT)


In [ ]:
# CELL 2 — ghi mã nguồn
JOB = 'j11_full.py'
ZIPNAME = 'j11_full_pilot'
PARAMS = {'PER_LABEL': 8, 'SEED': 909, 'HOURS': 10.5}
SRC = {'speech_gate.py': '"""Speech-anchored attention gating for Qwen2.5-Omni Thinker (single forward pass, no training, no re-encoding).\n\nQwen2.5-Omni gives every audio and video token a temporal rope id (25 ids per second, shared clock).\nGiven a time window [s0, s1] (seconds), we find the video tokens whose temporal patch overlaps the window and,\nfor every text query position AFTER the media block (question + assistant prefix), we add a bias to the attention\nlogits of video keys OUTSIDE the window:  hard = -inf (attention-level crop), soft = -beta.\nMedia tokens themselves attend as usual, so the visual and audio representations are unchanged; only what the\nquestion/answer positions read from is steered.\n"""\nimport torch\n\n\ndef media_layout(model, proc, inputs):\n    """Return position_ids (3,1,L), per-token time in seconds for video tokens, and index sets."""\n    ids = inputs[\'input_ids\']\n    am = inputs[\'attention_mask\']\n    afl = inputs[\'feature_attention_mask\'].sum(-1) if \'feature_attention_mask\' in inputs else None\n    spg = inputs.get(\'video_second_per_grid\')\n    pos, _ = model.get_rope_index(ids, None, inputs.get(\'video_grid_thw\'), am, True, afl, spg)\n    tok = proc.tokenizer\n    vid_id = tok.convert_tokens_to_ids(proc.video_token)\n    aud_id = tok.convert_tokens_to_ids(proc.audio_token)\n    is_v = (ids[0] == vid_id)\n    is_a = (ids[0] == aud_id)\n    t = pos[0, 0].float()\n    base = t[is_a].min() if is_a.any() else t[is_v].min()\n    sec = (t - base) / 25.0                       # temporal rope id -> seconds on the shared clock\n    last_media = int(torch.nonzero(is_v | is_a).max())\n    return pos, sec, is_v, is_a, last_media\n\n\ndef gated_mask(L, is_v, sec, last_media, window, mode, beta, dtype, device, spg=None, scope=\'all\', is_a=None, block_audio=False):\n    """scope=\'text\': only positions after the media block are gated (leaks through media tokens that already read the\n    out-of-window frames). scope=\'all\': no position except the out-of-window video tokens themselves may read them,\n    i.e. an attention-level crop that keeps positions, audio and the full token sequence."""\n    neg = torch.finfo(dtype).min\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    if mode != \'none\':\n        s0, s1 = window\n        width = float(spg) if spg is not None else 0.0   # a temporal patch covers [t, t + second_per_grid)\n        inside = (sec + width > s0) & (sec < s1)\n        out_v = is_v.to(device) & ~inside.to(device)\n        rows = (torch.arange(L, device=device) > last_media) if scope == \'text\' else ~out_v\n        bias = float(\'-inf\') if mode == \'hard\' else -float(beta)\n        m[rows.unsqueeze(1) & out_v.unsqueeze(0)] += bias\n    if block_audio:\n        # audio-blind decision: no non-audio position may read audio tokens (audio tokens still read each other)\n        aud = is_a.to(device)\n        m[(~aud).unsqueeze(1) & aud.unsqueeze(0)] = float(\'-inf\')\n    m = m.clamp(min=neg).to(dtype)\n    return m.view(1, 1, L, L)\n\n\ndef binding_mask(L, is_v, is_a, sec, dtype, device, chunk_s=2.0, delta=0, restrict=True, bidir=False, base=None):\n    """Synchronous audio-visual binding inside the media block (text rows are never changed).\n    Qwen2.5-Omni interleaves every 2-s chunk as [video_c, audio_c] under a causal mask, so\n      - video_c can never read audio_c (the sound of the SAME moment comes later in the sequence), and\n      - audio_c (and video_c) can read the other modality of ALL earlier moments.\n    restrict=True : a cross-modal edge (audio<->video) is kept only if the two chunks differ by <= delta\n    bidir=True    : additionally let video_c read audio_c\' for |c - c\'| <= delta even though it comes later (prefill only)\n    Within-modality attention and everything involving text tokens keep the original causal pattern."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device) if base is None else base.clone().float()\n    if base is None:\n        m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    ch = torch.floor(sec.to(device) / chunk_s)\n    near = (ch.unsqueeze(1) - ch.unsqueeze(0)).abs() <= delta\n    cross = (v.unsqueeze(1) & a.unsqueeze(0)) | (a.unsqueeze(1) & v.unsqueeze(0))\n    if restrict:\n        m[cross & ~near] = float(\'-inf\')\n    if bidir:\n        later = torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1)\n        m[(v.unsqueeze(1) & a.unsqueeze(0)) & near & later] = 0.0\n    neg = torch.finfo(dtype).min\n    return m.clamp(min=neg).to(dtype).view(1, 1, L, L)\n\n\ndef balance_mask(L, is_v, is_a, last_media, dtype, device, strength=1.0):\n    """Modality-balanced readout: question/answer rows get an additive bias on audio keys equal to\n    strength * log(n_video / n_audio), so that with equal content scores the two streams receive equal total attention\n    (Qwen2.5-Omni gives ~144 video tokens vs ~25 audio tokens per second)."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    b = strength * float(torch.log(v.sum().float() / a.sum().float().clamp(min=1)))\n    rows = torch.arange(L, device=device) > last_media\n    m[rows.unsqueeze(1) & a.unsqueeze(0)] += b\n    return m.clamp(min=torch.finfo(dtype).min).to(dtype).view(1, 1, L, L)\n\n\ndef bound_last_logits(model, proc, inputs, delta=0, restrict=True, bidir=False):\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    mask = binding_mask(L, is_v, is_a, sec, emb.dtype, emb.device, delta=delta, restrict=restrict, bidir=bidir)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True)\n    return out.logits[0, -1].float()\n\n\ndef gated_last_logits(model, proc, inputs, window=None, mode=\'none\', beta=0.0, use_audio_in_video=True, scope=\'all\', block_audio=False):\n    """Forward pass with explicit 3D rope ids and a 4D additive mask; returns last-position logits (float32)."""\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    spg = inputs.get(\'video_second_per_grid\')\n    spg = float(spg.reshape(-1)[0]) if spg is not None else None\n    mask = gated_mask(L, is_v, sec, last_media, window, mode, beta, emb.dtype, emb.device, spg, scope, is_a, block_audio)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=use_audio_in_video)\n    return out.logits[0, -1].float(), dict(n_video=int(is_v.sum()), n_audio=int(is_a.sum()), last_media=last_media,\n                                           video_sec_range=[float(sec[is_v].min()), float(sec[is_v].max())])\n', 'kaggle_rootcause.py': '"""Root-cause completion for SVHalluc TEMPORAL (Qwen2.5-Omni-3B). Usage:\n  python kaggle_rootcause.py ROOT OUT PER_LABEL SHARD_ID NUM_SHARDS\nPer item (TA/TF/CMTB), all scored with rotated option orders and summed per option content:\n  base                  native model                                                        -> reference answer\n  para1, para2          two paraphrases of the question                                    -> #12 question wording\n  prior                 black video + silent audio of the same length                     -> #13 content-free prior\n  vision_only           original video, audio removed                                     -> modality reliance\n  audio_only            black video + original audio                                      -> modality reliance\n  (CMTB) gate, audio_block, gate_audio, crop                                               -> #9/#10 per-error decomposition\n  hear (text) + whisper (analysis-only reference transcript of the speech window)          -> #4 speech content\n  hidden states (last token, mean of question rows) at several layers from the base run    -> #14 representation-action gap\n  (CMTB, base wrong and gate_audio right) activation patching sweep                        -> where the error enters\nShard 0 additionally runs:\n  synthetic text-only relation questions (no media)                                        -> #11 relation reasoning\n  eager attention of the question rows for the first 24 items                              -> #8 front-loading at scale\nLabels are used only for scoring. Whisper is an analysis tool only; it never feeds the model."""\nimport os, sys, json, re, subprocess, time, random\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nos.environ[\'FORCE_QWENVL_VIDEO_READER\'] = \'decord\'\nROOT, OUT = Path(sys.argv[1]), Path(sys.argv[2])\nPER_LABEL, SHARD, NSHARD = int(sys.argv[3]), int(sys.argv[4]), int(sys.argv[5])\nMAX_DUR = float(os.environ.get(\'LAB_MAX_DUR\', 42.0 if torch.cuda.device_count() >= 2 else 28.0))\nsys.path.insert(0, str(ROOT))\nimport av, librosa\nfrom huggingface_hub import hf_hub_download\n\nDATASET, REVISION = \'zcs15/SVHalluc\', \'211b4f14ab66ae9557b69e302d1f090e0e610839\'\nMODEL, MREV = \'Qwen/Qwen2.5-Omni-3B\', \'f75b40e3da2003cdd6e1829b1f420ca70797c34e\'\nBS = chr(92); FF = os.environ.get(\'FFMPEG_BIN\', \'ffmpeg\')\nLAYERS_KEEP = [6, 12, 15, 18, 21, 24, 27, 30, 33, 36]\n\n\ndef decode_audio(path):\n    chunks = []\n    with av.open(str(path)) as c:\n        rs = av.AudioResampler(format=\'fltp\', layout=\'mono\', rate=16000)\n        for fr in c.decode(audio=0):\n            for p in rs.resample(fr): chunks.append(p.to_ndarray().reshape(-1))\n        for p in rs.resample(None): chunks.append(p.to_ndarray().reshape(-1))\n    return np.concatenate(chunks).astype(\'float32\', copy=False)\n\n\n_orig = librosa.load\ndef _load(path, sr=None, offset=0.0, duration=None, **k):\n    if str(path).lower().endswith(\'.mp4\'):\n        a = decode_audio(path)\n        return a[int(sr * offset): None if duration is None else int(sr * (offset + duration))], sr\n    return _orig(path, sr=sr, offset=offset, duration=duration, **k)\nlibrosa.load = _load\n\nfrom lab_native_video import prepare_video, qwen_inputs, move_inputs, SYSTEM\nfrom speech_gate import media_layout, gated_mask\nfrom transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor\n\nCLIPS = OUT / \'clips\'; CLIPS.mkdir(parents=True, exist_ok=True)\nHID = OUT / \'hidden\'; HID.mkdir(parents=True, exist_ok=True)\nENC = [\'-c:v\', \'libx264\', \'-preset\', \'ultrafast\', \'-crf\', \'24\', \'-pix_fmt\', \'yuv420p\', \'-c:a\', \'aac\']\n\n\ndef fetch(name):\n    local = os.environ.get(\'LAB_LOCAL_DATA\')\n    if local and (Path(local) / name).exists(): return Path(local) / name\n    return Path(hf_hub_download(DATASET, name, repo_type=\'dataset\', revision=REVISION))\n\n\ndef ffmpeg(*args):\n    r = subprocess.run([FF, \'-hide_banner\', \'-y\', *args], capture_output=True)\n    if r.returncode != 0 and b\'fps_mode\' in r.stderr:      # older ffmpeg builds only know -vsync\n        r = subprocess.run([FF, \'-hide_banner\', \'-y\', *[\'-vsync\' if a == \'-fps_mode\' else a for a in args]], capture_output=True)\n    if r.returncode != 0: raise RuntimeError(r.stderr.decode(errors=\'replace\')[-400:])\n\n\ndef cut(src, start, end, tag):\n    dst = CLIPS / f\'{Path(src).stem}_{tag}.mp4\'\n    if not dst.exists():\n        ffmpeg(\'-ss\', f\'{start:.2f}\', \'-to\', f\'{end:.2f}\', \'-i\', str(src), \'-map\', \'0:v:0\', \'-map\', \'0:a:0\', \'-vf\', \'fps=30\', \'-fps_mode\', \'cfr\', *ENC,\n               \'-af\', \'aresample=async=1:first_pts=0\', str(dst))\n    return dst\n\n\ndef video_size(src):\n    with av.open(str(src)) as c:\n        s = c.streams.video[0]; return s.width, s.height\n\n\ndef blank_video(src, dur, keep_audio):\n    """black frames of the same size/length; either the original audio or digital silence."""\n    w, h = video_size(src)\n    dst = CLIPS / f\'{Path(src).stem}_{"blackA" if keep_audio else "prior"}.mp4\'\n    if not dst.exists():\n        if keep_audio:\n            ffmpeg(\'-f\', \'lavfi\', \'-i\', f\'color=black:s={w}x{h}:r=30:d={dur:.2f}\', \'-i\', str(src), \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', \'-shortest\', *ENC, str(dst))\n        else:\n            ffmpeg(\'-f\', \'lavfi\', \'-i\', f\'color=black:s={w}x{h}:r=30:d={dur:.2f}\', \'-f\', \'lavfi\', \'-i\', f\'anullsrc=r=16000:cl=mono\',\n                   \'-t\', f\'{dur:.2f}\', \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', *ENC, str(dst))\n    return dst\n\n\ndef speech_window(a):\n    hop = 800; n = len(a) // hop\n    db = 20 * np.log10(np.sqrt((a[:n * hop].reshape(n, hop) ** 2).mean(1) + 1e-12))\n    lo, hi = np.percentile(db, 10), np.percentile(db, 95)\n    act = db > lo + 0.5 * (hi - lo)\n    if act.sum() == 0: return None\n    t = (np.arange(n) + 0.5) * 0.05; idx = np.where(act)[0]\n    return float(t[idx[0]]), float(t[idx[-1]])\n\n\ndef parse_question(q):\n    parts = re.split(r\'\\s*\\(([ABC])\\)\\s*\', q)\n    return parts[0].strip(), [parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)]\n\n\ndef rotations(k): return [tuple((s + j) % k for j in range(k)) for s in range(k)]\ndef render(stem, texts, perm): return stem + \' \' + \' \'.join(f\'({L}) {texts[j]}\' for L, j in zip(\'ABC\', perm))\n\n\nPARA = {\'TA\': [\'Does the action that the speaker talks about happen in the video at the same moment the speaker is talking?\',\n               \'Look at the video only while the speech is playing. Is the narrated action visible during that time?\'],\n        \'TF\': [\'The speaker describes an action. Compared with the time the speaker is talking, does that action appear in the video earlier, at the same time, or later?\',\n               \'When does the narrated action happen in the video relative to the speech?\'],\n        \'CMTB\': [\'Which action is visible in the video while the person is speaking?\', \'During the speech, what is being done on screen?\']}\n\nMODEL_ = PROC = LETTERS = None\n\n\ndef load_model():\n    cache = os.environ.get(\'LAB_CACHE\')\n    kw = dict(revision=MREV, torch_dtype=torch.float16, attn_implementation=\'sdpa\', cache_dir=cache)\n    if os.environ.get(\'LAB_NF4\'):\n        from transformers import BitsAndBytesConfig\n        kw.update(device_map={\'\': 0}, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                  bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))\n    elif torch.cuda.device_count() >= 2: kw.update(device_map=\'balanced\', max_memory={0: \'10GiB\', 1: \'10GiB\'})\n    else: kw.update(device_map={\'\': 0})\n    m = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL, **kw).eval()\n    return m, Qwen2_5OmniProcessor.from_pretrained(MODEL, revision=MREV, cache_dir=cache)\n\n\ndef masked_forward(inp, mask_kind, window, record_layers=None, patch=None):\n    """mask_kind: plain | gate | audio_block | gate_audio. record_layers -> dict(layer -> (last_token, mean_question_rows, question_rows))."""\n    emb = MODEL_.get_input_embeddings().weight\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    L = inp[\'input_ids\'].shape[1]\n    spg = inp.get(\'video_second_per_grid\'); spg = float(spg.reshape(-1)[0]) if spg is not None else None\n    tg = mask_kind in (\'gate\', \'gate_audio\'); ab = mask_kind in (\'audio_block\', \'gate_audio\')\n    mask = gated_mask(L, is_v, sec, last_media, window, \'hard\' if tg else \'none\', 0.0, emb.dtype, emb.device, spg, \'all\', is_a, ab)\n    kw = {k: v for k, v in inp.items() if k != \'attention_mask\'}\n    rows_q = (torch.arange(L) > last_media)\n    store, hooks = {}, []\n    layers = MODEL_.model.layers\n    if record_layers or patch is not None:\n        for li in range(len(layers)):\n            def hook(mod, args, out, li=li):\n                h = out[0]\n                rq = rows_q.to(h.device)\n                if patch is not None and patch[0] == li:\n                    h = h.clone(); h[0, rq] = patch[1].to(h.device, h.dtype); out = (h,) + tuple(out[1:])\n                if record_layers and (li + 1) in record_layers:\n                    store[li + 1] = (h[0, -1].float().cpu(), h[0, rq].float().mean(0).cpu(), h[0, rq].detach().float().cpu())\n                return out\n            hooks.append(layers[li].register_forward_hook(hook))\n    try:\n        with torch.no_grad():\n            lg = MODEL_(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True).logits[0, -1].float()\n    finally:\n        for h in hooks: h.remove()\n    return lg, store\n\n\ndef scores(media, stem, texts, mode=\'av\', mask_kind=None, window=None):\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        inp, _ = qwen_inputs(PROC, render(stem, texts, perm), media, mode); inp = move_inputs(inp, MODEL_)\n        if mask_kind is None:\n            with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=(mode == \'av\')).logits[0, -1].float()\n        else:\n            lg, _ = masked_forward(inp, mask_kind, window)\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n        del inp\n    torch.cuda.empty_cache()\n    return tot.round(3).tolist()\n\n\nHEAR_Q = \'What action does the speaker say is being done? Answer with a short phrase (verb and object) only.\'\nWHISPER = None\n\n\ndef hear(media):\n    inp, _ = qwen_inputs(PROC, HEAR_Q, media, \'audio\'); inp = move_inputs(inp, MODEL_)\n    with torch.no_grad(): o = MODEL_.generate(**inp, max_new_tokens=16, do_sample=False)\n    return PROC.batch_decode(o[:, inp[\'input_ids\'].shape[1]:], skip_special_tokens=True)[0].strip().split(\'\\n\')[0][:120]\n\n\ndef whisper_text(a, s0, s1):\n    global WHISPER\n    if os.environ.get(\'LAB_NO_WHISPER\'): return None\n    if WHISPER is None:\n        from transformers import pipeline\n        WHISPER = pipeline(\'automatic-speech-recognition\', model=\'openai/whisper-small\', device=0, torch_dtype=torch.float16)\n    seg = a[int(s0 * 16000): int(s1 * 16000)]\n    return WHISPER({\'raw\': seg, \'sampling_rate\': 16000}, generate_kwargs={\'language\': \'en\'})[\'text\'].strip()\n\n\ndef patch_sweep(media, stem, texts, window):\n    """Original option order only. Returns plain/clean letter logits and forward/reverse patching per layer."""\n    inp, _ = qwen_inputs(PROC, render(stem, texts, tuple(range(len(texts)))), media, \'av\'); inp = move_inputs(inp, MODEL_)\n    k = len(texts); allL = list(range(1, len(MODEL_.model.layers) + 1))\n    lg_p, Sp = masked_forward(inp, \'plain\', window, record_layers=allL)\n    lg_c, Sc = masked_forward(inp, \'gate_audio\', window, record_layers=allL)\n    let = lambda lg: [round(float(lg[t]), 3) for t in LETTERS[:k]]\n    res = dict(plain=let(lg_p), clean=let(lg_c), fwd={}, rev={})\n    for li in list(range(0, len(MODEL_.model.layers), 3)) + [len(MODEL_.model.layers) - 1]:\n        res[\'fwd\'][li] = let(masked_forward(inp, \'plain\', window, patch=(li, Sc[li + 1][2]))[0])\n        res[\'rev\'][li] = let(masked_forward(inp, \'gate_audio\', window, patch=(li, Sp[li + 1][2]))[0])\n    del inp; torch.cuda.empty_cache()\n    return res\n\n\ndef eager_attention(media, stem, texts, window, dur):\n    """per layer: attention mass (summed over question rows, mean over heads) to video first/second half, inside/outside window, audio."""\n    inp, _ = qwen_inputs(PROC, render(stem, texts, tuple(range(len(texts)))), media, \'av\'); inp = move_inputs(inp, MODEL_)\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    L = inp[\'input_ids\'].shape[1]; rows_q = torch.arange(L) > last_media\n    sec_c, vi, ai = sec.cpu(), is_v.cpu(), is_a.cpu()\n    spg = float(inp[\'video_second_per_grid\'].reshape(-1)[0])\n    inside = (sec_c + spg > window[0]) & (sec_c < window[1])\n    groups = {\'v_first_half\': vi & (sec_c < dur / 2), \'v_second_half\': vi & (sec_c >= dur / 2), \'v_in\': vi & inside, \'v_out\': vi & ~inside, \'audio\': ai}\n    counts = {g: int(m.sum()) for g, m in groups.items()}\n    stats = {}\n    def mk(li):\n        def hook(mod, args, out):\n            w = out[1]\n            if w is None: return\n            dv = w.device; q = w[0][:, rows_q.to(dv)].float().sum(1).mean(0)\n            stats[li] = {g: round(float(q[m.to(dv)].sum()), 5) for g, m in groups.items()}\n            return (out[0], None)\n        return hook\n    MODEL_.set_attn_implementation(\'eager\')\n    hooks = [MODEL_.model.layers[li].self_attn.register_forward_hook(mk(li)) for li in range(len(MODEL_.model.layers))]\n    try:\n        with torch.no_grad(): MODEL_(**inp, use_audio_in_video=True)\n    finally:\n        for h in hooks: h.remove()\n        MODEL_.set_attn_implementation(\'sdpa\')\n    del inp; torch.cuda.empty_cache()\n    return dict(counts=counts, per_layer=[stats[li] for li in sorted(stats)])\n\n\ndef text_only_scores(stem, texts):\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        conv = [{\'role\': \'system\', \'content\': [{\'type\': \'text\', \'text\': SYSTEM}]}, {\'role\': \'user\', \'content\': [{\'type\': \'text\', \'text\': render(stem, texts, perm)}]}]\n        text = PROC.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)\n        inp = PROC(text=text, return_tensors=\'pt\').to(MODEL_.get_input_embeddings().weight.device)\n        with torch.no_grad(): lg = MODEL_(**inp).logits[0, -1].float()\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n    return tot.round(3).tolist()\n\n\nACTIONS = [\'pour the oil into the pan\', \'chop the onions\', \'add salt to the soup\', \'stir the sauce\', \'slice the tomatoes\',\n           \'crack the eggs into the bowl\', \'mash the potatoes\', \'spread butter on the bread\', \'fry the bacon\', \'mix the flour and water\']\n\n\nTENSES = {\'future\': \'I will {a}\', \'present\': \'Now I am going to {a} right now\', \'past\': \'I have already finished: {a}\', \'neutral\': \'{a}\'}\n\n\ndef synthetic_relation():\n    """relation (Past/Present/Future) x speech tense; text-only facts, no media. Tests reasoning and the tense shortcut."""\n    rng = random.Random(0); out = []\n    for tense, tmpl in TENSES.items():\n        for n in range(30):\n            act = rng.choice(ACTIONS); s0 = round(rng.uniform(10, 20), 1); sd = round(rng.uniform(2, 5), 1)\n            rel = n % 3                      # 0 Past, 1 Present, 2 Future\n            if rel == 1: e0 = round(s0 + rng.uniform(-0.5, 0.5), 1)\n            elif rel == 0: e0 = round(s0 - rng.uniform(6, 9), 1)\n            else: e0 = round(s0 + sd + rng.uniform(3, 9), 1)\n            ed = round(rng.uniform(2, 5), 1); said = tmpl.format(a=act)\n            facts = (f\'In a cooking video, a person says "{said}" from {s0} to {round(s0 + sd, 1)} seconds. \'\n                     f\'The action "{act}" is actually shown in the video from {e0} to {round(e0 + ed, 1)} seconds.\')\n            tf = text_only_scores(facts + \' Relative to the moment when the speech is heard, when does the narrated event occur in the video?\', [\'Past\', \'Present\', \'Future\'])\n            tf_plain = text_only_scores(facts + \' Is the action shown before, during, or after the speech?\', [\'Before\', \'During\', \'After\'])\n            ta = text_only_scores(facts + \' When the speech is heard, is the narrated event happening at the same time in the video?\', [\'Yes\', \'No\'])\n            out.append(dict(tense=tense, rel=rel, tf=tf, tf_plain=tf_plain, ta=ta, ta_gold=0 if rel == 1 else 1))\n    return out\n\n\ndef run_item(task, r, shard_attention):\n    orig = fetch(r[\'video_path\'].replace(BS, \'/\'))\n    full = cut(orig, 0, 10 ** 4, \'full\')\n    a = decode_audio(full); dur = len(a) / 16000\n    if dur > MAX_DUR: raise RuntimeError(\'skipped: longer than MAX_DUR\')\n    w = speech_window(a)\n    if w is None: raise RuntimeError(\'no speech activity\')\n    s0, s1 = w; win = (s0, s1)\n    media = prepare_video(full)\n    stem, texts = parse_question(r[\'question\']); k = len(texts)\n    rec = dict(dur=round(dur, 2), win=[round(s0, 2), round(s1, 2)])\n    # base + hidden states from the first option order\n    tot = np.zeros(k)\n    for n_perm, perm in enumerate(rotations(k)):\n        inp, _ = qwen_inputs(PROC, render(stem, texts, perm), media, \'av\'); inp = move_inputs(inp, MODEL_)\n        lg, store = masked_forward(inp, \'plain\', win, record_layers=LAYERS_KEEP if n_perm == 0 else None)\n        if n_perm == 0:\n            np.savez_compressed(HID / f\'h_{r["_idx"]}.npz\', last=np.stack([store[l][0].numpy() for l in LAYERS_KEEP]).astype(\'float16\'),\n                                qmean=np.stack([store[l][1].numpy() for l in LAYERS_KEEP]).astype(\'float16\'), layers=np.array(LAYERS_KEEP))\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n        del inp\n    rec[\'base\'] = tot.round(3).tolist()\n    rec[\'para1\'] = scores(media, PARA[task][0], texts)\n    rec[\'para2\'] = scores(media, PARA[task][1], texts)\n    rec[\'vision_only\'] = scores(media, stem, texts, mode=\'vision\')\n    rec[\'audio_only\'] = scores(prepare_video(blank_video(full, dur, True)), stem, texts)\n    rec[\'prior\'] = scores(prepare_video(blank_video(full, dur, False)), stem, texts)\n    rec[\'hear\'] = hear(media)\n    try: rec[\'whisper\'] = whisper_text(a, s0, s1)\n    except Exception as e: rec[\'whisper_error\'] = repr(e)[:200]\n    if task == \'CMTB\':\n        for mk in (\'gate\', \'audio_block\', \'gate_audio\'): rec[mk] = scores(media, stem, texts, mask_kind=mk, window=win)\n        rec[\'crop\'] = scores(prepare_video(cut(full, s0, s1, \'speech\')), stem, texts, mode=\'vision\')\n        g = \'ABC\'.index(r[\'answer\'])\n        if int(np.argmax(rec[\'base\'])) != g and int(np.argmax(rec[\'gate_audio\'])) == g:\n            rec[\'patching\'] = patch_sweep(media, stem, texts, win)\n    if shard_attention:\n        rec[\'attention\'] = eager_attention(media, stem, texts, win, dur)\n    return rec\n\n\ndef main():\n    global MODEL_, PROC, LETTERS\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    cells = {}\n    for i, r in enumerate(rows):\n        if r[\'data_type\'] == \'temporal\': cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n    items = []\n    for key in sorted(cells):\n        pool = cells[key]; random.Random(1).shuffle(pool); items += pool[:PER_LABEL]\n    items.sort(key=lambda x: x[0])\n    mine = [x for k, x in enumerate(items) if k % NSHARD == SHARD]\n    if os.environ.get(\'LAB_ONLY_IDS\'):\n        want = {int(v) for v in os.environ[\'LAB_ONLY_IDS\'].split(\',\')}; mine = [x for x in enumerate(rows) if x[0] in want]\n    res = OUT / f\'rootcause_shard{SHARD}of{NSHARD}.jsonl\'\n    done = {json.loads(l)[\'idx\'] for l in res.read_text().splitlines()} if res.exists() else set()\n    MODEL_, PROC = load_model(); LETTERS = [PROC.tokenizer.convert_tokens_to_ids(L) for L in \'ABC\']\n    print(\'model ready; items\', len(mine), \'done\', len(done), flush=True)\n    syn = OUT / \'synthetic_relation.json\'\n    if SHARD == 0 and not syn.exists() and not os.environ.get(\'LAB_NO_SYNTH\'):\n        syn.write_text(json.dumps(synthetic_relation()), encoding=\'utf-8\'); print(\'synthetic relation done\', flush=True)\n    n_att = 0\n    for n, (i, r) in enumerate(mine):\n        if i in done: continue\n        t0 = time.time(); task = r[\'task_type_abbr\']\n        rec = dict(idx=i, task=task, gold=\'ABC\'.index(r[\'answer\']))\n        try:\n            r = dict(r); r[\'_idx\'] = i\n            att = SHARD == 0 and n_att < 24\n            rec.update(run_item(task, r, att)); n_att += int(att); rec[\'sec\'] = round(time.time() - t0)\n        except Exception as e:\n            rec[\'error\'] = repr(e)[:300]\n        torch.cuda.empty_cache()\n        with res.open(\'a\') as f: f.write(json.dumps(rec) + \'\\n\')\n        if \'error\' in rec: print(n + 1, \'/\', len(mine), task, i, rec[\'error\'][:60], flush=True); continue\n        g = rec[\'gold\']; ok = lambda c: \'Y\' if c in rec and int(np.argmax(rec[c])) == g else \'-\'\n        print(n + 1, \'/\', len(mine), task, i, \'base\', ok(\'base\'), \'p1\', ok(\'para1\'), \'p2\', ok(\'para2\'), \'vis\', ok(\'vision_only\'), \'aud\', ok(\'audio_only\'),\n              \'GA\', ok(\'gate_audio\'), \'patch\' if \'patching\' in rec else \'\', rec[\'sec\'], \'s\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'job_utils.py': '"""Shared helpers for the diagnostic notebooks (exec\'d in the kaggle_rootcause.py namespace after the model is loaded).\nLabels are only ever used for scoring/analysis; nothing here feeds a label to the model."""\nimport json, random, time\nfrom pathlib import Path\nimport numpy as np, torch, soundfile as sf\n\nL4 = [PROC.tokenizer.convert_tokens_to_ids(c) for c in \'ABCD\']\nSR = 16000\n\n\n# ---------------------------------------------------------------- items\ndef sample_items(tasks, per_label, seed, max_dur=None):\n    """Balanced per (task, gold letter); round-robin across cells so a deadline cut keeps the sample balanced."""\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    cells = {}\n    for i, r in enumerate(rows):\n        if r[\'data_type\'] == \'temporal\' and r[\'task_type_abbr\'] in tasks: cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n    pools = []\n    for key in sorted(cells):\n        pool = cells[key]; random.Random(seed).shuffle(pool); pools.append(pool[:per_label])\n    items = []\n    for k in range(max(len(p) for p in pools)):\n        for p in pools:\n            if k < len(p): items.append(p[k])\n    return items\n\n\ndef all_temporal_items():\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    return [(i, r) for i, r in enumerate(rows) if r[\'data_type\'] == \'temporal\']\n\n\ndef prep_clip(r, max_dur=None):\n    src = cut(fetch(r[\'video_path\'].replace(BS, \'/\')), 0, 10 ** 4, \'full\')\n    a = decode_audio(src); dur = len(a) / SR\n    if max_dur is not None and dur > max_dur: raise RuntimeError(\'long\')\n    w = speech_window(a)\n    if w is None: raise RuntimeError(\'no speech activity\')\n    return src, a, dur, w\n\n\ndef speech_first(w, dur): return (w[0] + w[1]) / 2 < dur / 2\n\n\n# ---------------------------------------------------------------- scoring (up to 4 options, rotated orders summed per content)\ndef prompt_of(stem, texts, perm): return stem + \' \' + \' \'.join(f\'({L}) {texts[j]}\' for L, j in zip(\'ABCD\', perm))\n\n\ndef score(media, stem, texts, mode=\'av\', fwd=None, on_logits=None):\n    """fwd(inp) -> last-token logits (default: plain forward). on_logits(perm, logits) is called for every order."""\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        inp, _ = qwen_inputs(PROC, prompt_of(stem, texts, perm), media, mode); inp = move_inputs(inp, MODEL_)\n        if fwd is None:\n            with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=(mode == \'av\')).logits[0, -1].float()\n        else: lg = fwd(inp)\n        if on_logits is not None: on_logits(perm, lg)\n        lp = torch.log_softmax(lg[L4[:k]].cpu(), -1).numpy()\n        for p_, j in enumerate(perm): tot[j] += lp[p_]\n        del inp\n    torch.cuda.empty_cache()\n    return tot.round(3).tolist()\n\n\ndef yes_minus_no(media, question, mode=\'vision\'):\n    s = score(media, question, [\'Yes\', \'No\'], mode=mode); return round(s[0] - s[1], 3)\n\n\ndef layer_out(out):\n    h = out[0] if isinstance(out, tuple) else out\n    return h[0] if h.dim() == 3 else h\n\n\ndef forward_record(inp, layers, groups, use_audio=True):\n    """plain forward; for each recorded layer (1-based output index) the mean hidden state of every boolean token group."""\n    store, hooks = {}, []\n    for li in layers:\n        def hook(mod, args, out, li=li):\n            h = layer_out(out)\n            store[li] = {g: h[m.to(h.device)].float().mean(0).cpu() for g, m in groups.items() if bool(m.any())}\n        hooks.append(MODEL_.model.layers[li - 1].register_forward_hook(hook))\n    try:\n        with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=use_audio).logits[0, -1].float()\n    finally:\n        for h in hooks: h.remove()\n    return lg, store\n\n\ndef token_groups(inp, dur, win):\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    sec, is_v, is_a = sec.cpu(), is_v.cpu(), is_a.cpu()\n    L = inp[\'input_ids\'].shape[1]; idx = torch.arange(L)\n    inside = (sec >= win[0] - 1.0) & (sec <= win[1] + 1.0)\n    g = {\'last\': idx == L - 1, \'q\': idx > last_media,\n         \'v_first\': is_v & (sec < dur / 2), \'v_second\': is_v & (sec >= dur / 2),\n         \'a_speech\': is_a & inside, \'a_rest\': is_a & ~inside}\n    return g, (pos, sec, is_v, is_a, last_media)\n\n\n# ---------------------------------------------------------------- clip variants (ffmpeg, cached in CLIPS)\ndef with_audio(src, audio, tag):\n    w = CLIPS / f\'{Path(src).stem}_{tag}.wav\'; sf.write(w, audio, SR)\n    dst = CLIPS / f\'{Path(src).stem}_{tag}.mp4\'\n    if not dst.exists(): ffmpeg(\'-i\', str(src), \'-i\', str(w), \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', \'-c:v\', \'copy\', \'-c:a\', \'aac\', \'-shortest\', str(dst))\n    return dst\n\n\ndef concat_clips(parts, tag, audio_from=None):\n    dst = CLIPS / f\'{Path(parts[0]).stem}_{tag}.mp4\'\n    if dst.exists(): return dst\n    ins = []\n    for p in parts: ins += [\'-i\', str(p)]\n    n = len(parts)\n    if audio_from is None:\n        fc = \'\'.join(f\'[{k}:v:0][{k}:a:0]\' for k in range(n)) + f\'concat=n={n}:v=1:a=1[v][a]\'\n        ffmpeg(*ins, \'-filter_complex\', fc, \'-map\', \'[v]\', \'-map\', \'[a]\', *ENC, str(dst))\n    else:\n        fc = \'\'.join(f\'[{k}:v:0]\' for k in range(n)) + f\'concat=n={n}:v=1:a=0[v]\'\n        ffmpeg(*ins, \'-i\', str(audio_from), \'-filter_complex\', fc, \'-map\', \'[v]\', \'-map\', f\'{n}:a:0\', *ENC, \'-shortest\', str(dst))\n    return dst\n\n\ndef halves(src, dur):\n    return cut(src, 0, dur / 2, \'h1\'), cut(src, dur / 2, dur, \'h2\')\n\n\ndef pink(n, seed):\n    X = np.fft.rfft(np.random.default_rng(seed).standard_normal(n)); f = np.arange(len(X), dtype=\'float64\'); f[0] = 1\n    x = np.fft.irfft(X / np.sqrt(f), n); return (x / (x.std() + 1e-12)).astype(\'float32\')\n\n\ndef noisy_audio(a, w, snr_db=10.0, seed=0):\n    """pink noise over the whole track at snr_db below the speech RMS: removes the benchmark\'s digital silence."""\n    s = a[int(w[0] * SR): int(w[1] * SR)]; ps = float(np.sqrt(np.mean(s ** 2) + 1e-12))\n    return np.clip(a + pink(len(a), seed) * ps * 10 ** (-snr_db / 20), -1, 1).astype(\'float32\')\n\n\n# ---------------------------------------------------------------- loop with resume + deadline\ndef run_loop(items, out_name, fn, show=(), min_left=1200):\n    items = items[:int(globals().get(\'LIMIT\', 10 ** 9))]          # LIMIT is set only by the smoke test\n    res = OUT / out_name\n    done = {json.loads(l)[\'idx\'] for l in res.read_text().splitlines() if l.strip()} if res.exists() else set()\n    print(\'items\', len(items), \'already done\', len(done), flush=True)\n    group_stats = {}\n    for n, (i, r) in enumerate(items):\n        if i in done: continue\n        if time_left() < min_left: print(\'deadline reached: stopping cleanly\', flush=True); break\n        t0 = time.time(); rec = dict(idx=i, task=r[\'task_type_abbr\'], gold=\'ABCD\'.index(r[\'answer\']))\n        try:\n            rec.update(fn(i, r)); rec[\'sec\'] = round(time.time() - t0)\n        except torch.cuda.OutOfMemoryError:\n            rec[\'error\'] = \'OOM\'\n        except Exception as e:\n            rec[\'error\'] = repr(e)[:300]\n        torch.cuda.empty_cache()\n        with res.open(\'a\') as f: f.write(json.dumps(rec) + \'\\n\')\n        if \'error\' in rec: print(n + 1, len(items), rec[\'task\'], i, \'ERR\', rec[\'error\'][:80], flush=True); continue\n        g = rec[\'gold\']\n        marks = \' \'.join(f\'{c}={"Y" if int(np.argmax(rec[c])) == g else "-"}\' for c in show if isinstance(rec.get(c), list))\n        print(n + 1, len(items), rec[\'task\'], i, marks, rec[\'sec\'], \'s\', flush=True)\n        if globals().get(\'PROGRESS_GROUPS\', False):\n            key = (rec[\'task\'], int(g))\n            z = group_stats.setdefault(key, {c: [0, 0] for c in show})\n            for c in show:\n                if isinstance(rec.get(c), list):\n                    z[c][1] += 1\n                    z[c][0] += int(np.argmax(rec[c]) == g)\n            parts = []\n            for (tt, gg), zz in sorted(group_stats.items()):\n                parts.append(f\'{tt}/{chr(65 + gg)} \' + \' \'.join(f\'{c}={zz[c][0]}/{zz[c][1]}\' for c in show if c in zz))\n            print(\'GROUP_PROGRESS \' + \' | \'.join(parts), flush=True)\n    return [json.loads(l) for l in res.read_text().splitlines() if l.strip()] if res.exists() else []\n\n\ndef acc(R, cond, task=None, expect=None):\n    """accuracy (%) and n of condition `cond`; expect(rec) gives the expected index (default: gold)."""\n    v = [int(np.argmax(r[cond])) == (expect(r) if expect else r[\'gold\']) for r in R\n         if \'error\' not in r and isinstance(r.get(cond), list) and (task is None or r[\'task\'] == task) and (expect is None or expect(r) is not None)]\n    return (round(100 * float(np.mean(v)), 1) if v else None, len(v))\n', 'nb_runner.py': '"""Load Qwen2.5-Omni-3B once into the namespace of kaggle_rootcause.py, then exec job_utils.py and one job file in it.\nUsage: python nb_runner.py ROOT OUT JOB.py \'{"PARAM": value, "HOURS": 10.5}\'\nThe job sees every helper of kaggle_rootcause.py (cut, fetch, scores, masked_forward, hear, whisper_text, ...), the model\n(MODEL_, PROC, LETTERS), OUT, the params as globals, and time_left() (seconds before the self-imposed deadline)."""\nimport sys, os, time, json\nfrom pathlib import Path\n\nROOT, OUT, JOB = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])\nPARAMS = json.loads(sys.argv[4]) if len(sys.argv) > 4 else {}\nt_start = time.time()\nsys.argv = [\'kaggle_rootcause.py\', str(ROOT), str(OUT), \'1\', \'0\', \'1\']\nNS = {\'__name__\': \'rc_namespace\', \'__file__\': str(ROOT / \'kaggle_rootcause.py\')}\nexec(compile((ROOT / \'kaggle_rootcause.py\').read_text(encoding=\'utf-8\'), \'kaggle_rootcause.py\', \'exec\'), NS)\nNS[\'MODEL_\'], NS[\'PROC\'] = NS[\'load_model\']()\nNS[\'LETTERS\'] = [NS[\'PROC\'].tokenizer.convert_tokens_to_ids(L) for L in \'ABC\']\nhours = float(PARAMS.pop(\'HOURS\', 10.5))\nNS.update(OUT=OUT, ROOT=ROOT, DEADLINE=t_start + hours * 3600, **PARAMS)\nNS[\'time_left\'] = lambda: NS[\'DEADLINE\'] - time.time()\nprint(\'model ready in\', round(time.time() - t_start), \'s; params\', PARAMS, \'; deadline in\', hours, \'h\', flush=True)\nexec(compile((ROOT / \'job_utils.py\').read_text(encoding=\'utf-8\'), \'job_utils.py\', \'exec\'), NS)\nexec(compile(JOB.read_text(encoding=\'utf-8\'), JOB.name, \'exec\'), NS)\nprint(\'RESULT\', json.dumps(NS.get(\'RESULT\'), default=str, indent=1), flush=True)\n', 'j11_full.py': '"""J11 — the full automatic method (human-inspired), every step recorded for failure analysis. Uses J8 helpers.\n 1 ORIENT   anchor = the question\'s attention to the AUDIO tokens per 2-s chunk, PMI vs a neutral question (alpha 2)\n 2 REHEARSE inner speech: the model itself says, in one short phrase, what it hears (any sound; the action if a person speaks)\n 3 LOOK     for each window (before / during / after the anchor) and each content (heard phrase + content options):\n            Yes/No "is this shown?" with the question rows\' audio blocked in every layer and video restricted to the window in\n            layers 13-27 (attention routing; the full clip stays in context)\n 4 REMEMBER source-separated episodic timeline: the narration declared once and marked as heard-only; per window only\n            "speech: yes/no" and what is SEEN (each content in its best window; each window keeps its top-contrast option)\n 5 REASON   (a) plain one-step answer from the timeline   (b) step-by-step reasoning, answer parsed from the reasoning, the\n            three option orders vote\nReference conditions: base; acar = single-pass in-model anchor + audio blocked (J8 best).\nWrites OUT/j11_full.jsonl."""\nimport json, re, numpy as np, torch\nexec((ROOT / \'j8_acar.py\').read_text(encoding=\'utf-8\').split(\'items = sample_items\')[0], globals())\n\nPER_LABEL = int(globals().get(\'PER_LABEL\', 8)); SEED = int(globals().get(\'SEED\', 909))\nINNER_Q = \'Listen to the audio. If a person speaks, give only the action they talk about (verb and object). Otherwise name the sound you hear. Answer with one short phrase.\'\nLOOK_T = \'Does the video show someone doing the following: "{c}"?\'\nNON_CONTENT = {\'yes\', \'no\', \'past\', \'present\', \'future\', \'before\', \'during\', \'after\', \'true\', \'false\'}\nCOT = \'\\nThink step by step about which part of the timeline the question asks about, then give the answer.\'\nSTOP = re.compile(r\'\\n\\s*(Human|User|Question)\\s*:|If you have any other questions\', re.I)\n\n\ndef chat_text(prompt):\n    conv = [{\'role\': \'system\', \'content\': [{\'type\': \'text\', \'text\': SYSTEM}]}, {\'role\': \'user\', \'content\': [{\'type\': \'text\', \'text\': prompt}]}]\n    return PROC.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)\n\n\ndef gen(prompt, n=160):\n    inp = PROC(text=chat_text(prompt), return_tensors=\'pt\').to(MODEL_.get_input_embeddings().weight.device)\n    with torch.no_grad(): o = MODEL_.generate(**inp, max_new_tokens=n, do_sample=False)\n    return PROC.batch_decode(o[:, inp[\'input_ids\'].shape[1]:], skip_special_tokens=True)[0].strip()\n\n\ndef inner_speech(media):\n    inp, _ = qwen_inputs(PROC, INNER_Q, media, \'audio\'); inp = move_inputs(inp, MODEL_)\n    with torch.no_grad(): o = MODEL_.generate(**inp, max_new_tokens=20, do_sample=False)\n    t = PROC.batch_decode(o[:, inp[\'input_ids\'].shape[1]:], skip_special_tokens=True)[0].strip().split(\'\\n\')[0]\n    t = re.split(r\'(?<=[.!?])\\s\', STOP.split(t)[0])[0]                              # first sentence only, no run-on\n    del inp; torch.cuda.empty_cache(); return t.strip().strip(\'."\')[:100]\n\n\ndef grow_window(rel, raw, frac=0.3):\n    """seed = the PMI peak (WHICH event), extent = contiguous chunks whose raw attention >= frac * max (HOW LONG it lasts)."""\n    seed = int(np.argmax(rel)); on = raw >= frac * raw.max(); on[seed] = True; lo = hi = seed\n    while lo - 1 >= 0 and on[lo - 1]: lo -= 1\n    while hi + 1 < len(on) and on[hi + 1]: hi += 1\n    return lo, hi + 1\n\n\ndef look_mask(lay, t0, t1):\n    rr = torch.where(lay.rows)[0]; A = torch.where(lay.a)[0]\n    na = lay.m.clone(); na[rr.unsqueeze(1), A.unsqueeze(0)] = float(\'-inf\')\n    win = window_mask(lay, t0, t1, True)\n    return {li: (win if li in ROUTE else na) for li in range(NL)}\n\n\ndef parse_answer(txt, shown):\n    txt = STOP.split(txt)[0]\n    m = list(re.finditer(r\'answer\\s+is\\W{0,4}\\(?([A-D])\\b\', txt, re.I)) or list(re.finditer(r\'\\(([A-D])\\)\', txt))\n    if m:\n        j = \'ABCD\'.index(m[-1].group(1).upper())\n        if j < len(shown): return j\n    tail = txt[txt.lower().rfind(\'answer\'):] if \'answer\' in txt.lower() else txt\n    hits = [j for j, t in enumerate(shown) if re.search(r\'\\b\' + re.escape(t.lower()) + r\'\\b\', tail.lower())]\n    return hits[-1] if len(hits) == 1 else None\n\n\ndef item(i, r):\n    src, a, dur, w = prep_clip(r, MAXD)\n    media = prepare_video(src); stem, texts = parse_question(r[\'question\']); k = len(texts)\n    rec = dict(dur=round(dur, 2), win=[round(w[0], 2), round(w[1], 2)], speech_first=speech_first(w, dur))\n    rec[\'base\'] = score(media, stem, texts)\n    # 1 orient\n    P = profiles(media, prompt_of(stem, texts, tuple(range(k)))); P0 = profiles(media, NEUTRAL); rel = pmi(P[\'a\'], P0[\'a\'])\n    s0, s1 = grow_window(rel, P[\'a\']); T0, T1 = 2.0 * s0, min(dur, 2.0 * s1); rec[\'anchor\'] = [T0, T1]; rec[\'rel_a\'] = np.round(rel, 4).tolist()\n    rec[\'acar\'] = score(media, stem, texts, fwd=lambda inp: (lambda lay: forward(inp, lay, bias_mask(lay, rel, 4.0, (\'v\',), True))[0])(Lay(inp)))\n    # 2 rehearse\n    claim = inner_speech(media); rec[\'claim\'] = claim\n    contents = [claim] + [t for t in texts if len(re.findall(r"[A-Za-z\']+", t)) >= 2 and t.lower().strip() not in NON_CONTENT and t != claim]\n    # 3 look\n    segs = [(n, t0, t1) for n, t0, t1 in ((\'before the speech\', 0.0, T0), (\'while the speech is heard\', T0, T1), (\'after the speech\', T1, dur)) if t1 - t0 >= 1.0]\n    ver = {}\n    for c in contents:\n        ver[c] = []\n        for _, t0, t1 in segs:\n            s = score(media, LOOK_T.format(c=c), [\'Yes\', \'No\'], fwd=lambda inp, t0=t0, t1=t1: (lambda lay: forward(inp, lay, look_mask(lay, t0, t1))[0])(Lay(inp)))\n            ver[c].append(round(s[0] - s[1], 3))\n    rec[\'segs\'] = [[n, round(t0, 2), round(t1, 2)] for n, t0, t1 in segs]; rec[\'verify\'] = ver\n    # 4 remember\n    seen = {j: [] for j in range(len(segs))}\n    if max(ver[claim]) > 0: seen[int(np.argmax(ver[claim]))].append(claim)          # only if actually seen somewhere\n    best = {j: None for j in range(len(segs))}\n    for t in contents[1:]:\n        v = ver[t]; j = int(np.argmax(v)); con = v[j] - max([x for jj, x in enumerate(v) if jj != j] or [-1e9])\n        if v[j] <= 0: continue                                                       # never seen -> not in the timeline\n        if best[j] is None or con > best[j][0]: best[j] = (con, t)\n    for j in best:\n        if best[j]: seen[j].append(best[j][1])\n    tl = (f\'Narration (what the speaker talks about; this is only heard, it is NOT evidence of what is shown): "{claim}"\\n\'\n          \'Timeline of the video (SEEN = checked on the video with the sound off):\\n\' +\n          \'\\n\'.join(f\'- {t0:.1f}-{t1:.1f} s ({n}): speech: {"yes" if n == "while the speech is heard" else "no"}; SEEN: \' +\n                    (\', \'.join(f\'"{x}"\' for x in seen[j]) or \'none of the listed actions\') for j, (n, t0, t1) in enumerate(segs)) + \'\\n\')\n    rec[\'timeline\'] = tl; q = tl + \'Answer using only the timeline. \' + stem\n    # 5 reason\n    rec[\'plain\'] = text_only_scores(q, texts)\n    votes = np.zeros(k); outs = []\n    for perm in rotations(k):\n        txt = gen(render(q, texts, perm) + COT); p = parse_answer(txt, [texts[j] for j in perm]); outs.append(STOP.split(txt)[0][:300])\n        if p is not None: votes[perm[p]] += 1\n    rec[\'cot\'] = (votes + 1e-3 * np.exp(np.array(rec[\'plain\']))).tolist(); rec[\'cot_texts\'] = outs\n    return rec\n\n\nitems = sample_items((\'TA\', \'TF\', \'CMTB\'), PER_LABEL, seed=SEED)\nC = (\'base\', \'acar\', \'plain\', \'cot\')\nR = run_loop(items, \'j11_full.jsonl\', item, show=C)\nOK = [r for r in R if \'error\' not in r]\nRESULT = {t: {c: acc(OK, c, t)[0] for c in C} | {\'n\': sum(r[\'task\'] == t for r in OK)} for t in (\'TA\', \'TF\', \'CMTB\')}\nRESULT[\'ALL\'] = {c: acc(OK, c)[0] for c in C}\nprint(RESULT, flush=True)\n', 'j8_acar.py': '"""J8 — QUESTION-GUIDED TEMPORAL ATTENTION RE-ALLOCATION (training-free, inside the model; no ASR, no text pipeline).\nGoal (team): for any text question, the video LM itself should focus on the part of the video - frames AND audio - that is\nneeded to answer, and the rest should contribute almost nothing.\nPass 1 (prefill, eager): relevance of every 2-s chunk c = attention of the question/answer rows to the tokens of c, averaged\nover all heads of layers 12-35 (label-free), measured separately on video tokens (v), audio tokens (a) and both (av).\nSalience correction (PMI): the same profile for a neutral question ("What is happening in the video?") is divided out,\n  rel(c) = p(c|q) * (p(c|q) / p(c|neutral))^ALPHA\nso chunks that are always salient (e.g. speech) only win when the question is about them (J9: raw attention always lands on\nthe speech even when the question asks about a bell).\nPass 2: in layers 13-27 the question/answer rows get LAMBDA*log(rel(c)/max rel) on the tokens of chunk c.\nConditions (rotated options):\n  base\n  oracle_route        hard window from the clip\'s audio activity, video only + audio blocked (upper bound for SVHalluc)\n  acar_a_l4_ab        audio-attention anchor (raw), re-weight video, audio blocked                  (speech-anchor design)\n  acar_a_pmi_l4_ab    same with PMI anchor\n  qtar_av_pmi_l1      GENERAL: av relevance with PMI, re-weight video AND audio tokens, lambda 1      (team proposal)\n  qtar_av_pmi_l4      same, lambda 4\n  qtar_av_pmi_l4_sg   same + source gating: audio tokens get the bias too but the video re-weighting is applied alone in the\n                      upper half of the routed layers (late layers read the selected scene with the sound off)\n  (TA/TF) look        "Is the action that the speaker talks about shown in the video?" with the question rows reading all audio\n                      but video only before / during / after the anchor window (analysis + rule readout, reported separately)\nAnalysis only: IoU of every relevance profile\'s window with the true speech chunks. Writes OUT/j8_acar.jsonl."""\nimport numpy as np, torch\n\nPER_LABEL = int(globals().get(\'PER_LABEL\', 8))\nSEED = int(globals().get(\'SEED\', 909))\nMAXD = float(globals().get(\'J8_MAX_DUR\', 34.0))\nALPHA = float(globals().get(\'ALPHA\', 2.0))\nNL = len(MODEL_.model.layers)\nROUTE = list(range(13, 28))\nREL_LAYERS = list(range(12, NL))\nNEUTRAL = \'What is happening in the video?\'\nLOOK_Q = \'Is the action that the speaker talks about shown in the video?\'\n\n\nclass Lay:\n    def __init__(self, inp):\n        emb = MODEL_.get_input_embeddings().weight; self.dev, self.dtype = emb.device, emb.dtype\n        self.pos, sec, v, a, self.last_media = media_layout(MODEL_, PROC, inp)\n        self.L = inp[\'input_ids\'].shape[1]; sec, v, a = sec.cpu(), v.cpu(), a.cpu()\n        self.v, self.a = v, a; self.nc = int(sec[v | a].max() // 2) + 1\n        self.chunk = torch.clamp((sec // 2).long(), 0, self.nc - 1)\n        self.rows = torch.arange(self.L) > self.last_media\n        m = torch.zeros(self.L, self.L); self.m = m.masked_fill(torch.triu(torch.ones(self.L, self.L, dtype=torch.bool), 1), float(\'-inf\'))\n\n    def t(self, m): return m.clamp(min=torch.finfo(self.dtype).min).to(self.dtype).to(self.dev).view(1, 1, self.L, self.L)\n\n\ndef forward(inp, lay, special=None, layers=ROUTE, record=None):\n    """special: mask used in `layers` (a dict {layer: mask} is also accepted); record: layers whose question-row attention\n    mass per chunk on video and audio tokens is returned (eager)."""\n    hooks, rec = [], {}\n    if special is not None:\n        per = special if isinstance(special, dict) else {li: special for li in layers}\n        per_t = {}\n        for li, mk in per.items():\n            key = id(mk)\n            if key not in per_t: per_t[key] = lay.t(mk)\n            sp = per_t[key]\n            def pre(mod, args, kwargs, sp=sp):\n                am_ = kwargs.get(\'attention_mask\'); kwargs[\'attention_mask\'] = sp.to(am_.device if am_ is not None else sp.device); return args, kwargs\n            hooks.append(MODEL_.model.layers[li].self_attn.register_forward_pre_hook(pre, with_kwargs=True))\n    if record:\n        MODEL_.set_attn_implementation(\'eager\')\n        Ga = torch.zeros(lay.L, lay.nc); Ga[torch.where(lay.a)[0], lay.chunk[lay.a]] = 1\n        Gv = torch.zeros(lay.L, lay.nc); Gv[torch.where(lay.v)[0], lay.chunk[lay.v]] = 1\n        for li in record:\n            def hk(mod, args, out, li=li):\n                w = out[1]\n                if w is None: return\n                q = w[0][:, lay.rows.to(w.device), :].float().mean(1)                  # [H, L]\n                rec[li] = ((q @ Ga.to(w.device)).mean(0).cpu(), (q @ Gv.to(w.device)).mean(0).cpu())\n                return (out[0], None) + tuple(out[2:])\n            hooks.append(MODEL_.model.layers[li].self_attn.register_forward_hook(hk))\n    kw = {k: v for k, v in inp.items() if k != \'attention_mask\'}\n    try:\n        with torch.no_grad():\n            lg = MODEL_(**kw, attention_mask=lay.t(lay.m), position_ids=lay.pos.to(lay.dev), use_audio_in_video=True).logits[0, -1].float()\n    finally:\n        for h in hooks: h.remove()\n        if record: MODEL_.set_attn_implementation(\'sdpa\')\n    return lg, rec\n\n\ndef profiles(media, prompt):\n    """relevance per chunk from the question rows: {\'a\',\'v\',\'av\'} normalised to sum 1."""\n    inp, _ = qwen_inputs(PROC, prompt, media, \'av\'); inp = move_inputs(inp, MODEL_)\n    lay = Lay(inp); _, rec = forward(inp, lay, record=REL_LAYERS)\n    a = torch.stack([rec[li][0] for li in REL_LAYERS]).mean(0).numpy(); v = torch.stack([rec[li][1] for li in REL_LAYERS]).mean(0).numpy()\n    del inp; torch.cuda.empty_cache()\n    n = lambda x: np.maximum(x / x.sum(), 1e-6)\n    return {\'a\': n(a), \'v\': n(v), \'av\': n(n(a) + n(v))}      # each modality normalised first: ~144 video vs 25 audio tokens/s\n\n\ndef pmi(p, p0): r = p * (p / np.maximum(p0, 1e-6)) ** ALPHA; return np.maximum(r / r.sum(), 1e-6)\n\n\ndef bias_mask(lay, rel, lam, on=(\'v\',), block_audio=False):\n    m = lay.m.clone(); r = torch.where(lay.rows)[0]\n    bias = lam * torch.log(torch.as_tensor(rel, dtype=torch.float32) / float(max(rel)))\n    for mod in on:\n        cols = torch.where(lay.v if mod == \'v\' else lay.a)[0]\n        m[r.unsqueeze(1), cols.unsqueeze(0)] += bias[lay.chunk[cols]].unsqueeze(0)\n    if block_audio: m[r.unsqueeze(1), torch.where(lay.a)[0].unsqueeze(0)] = float(\'-inf\')\n    return m\n\n\ndef window_mask(lay, t0, t1, block_audio):\n    m = lay.m.clone(); c0, c1 = int(t0 // 2), max(int(t0 // 2) + 1, int(np.ceil(t1 / 2)))\n    out_v = lay.v & ((lay.chunk < c0) | (lay.chunk >= c1)); r = torch.where(lay.rows)[0]\n    m[r.unsqueeze(1), torch.where(out_v)[0].unsqueeze(0)] = float(\'-inf\')\n    if block_audio: m[r.unsqueeze(1), torch.where(lay.a)[0].unsqueeze(0)] = float(\'-inf\')\n    return m\n\n\ndef top_window(p):\n    on = list(p > p.mean()) + [False]; best = (0, 0, 0); st = None\n    for k, x in enumerate(on):\n        if x and st is None: st = k\n        if not x and st is not None:\n            if k - st > best[0]: best = (k - st, st, k)\n            st = None\n    return best[1], best[2]\n\n\ndef chunk_activity(a, n):\n    hop = 800; k = len(a) // hop\n    db = 20 * np.log10(np.sqrt((a[:k * hop].reshape(k, hop) ** 2).mean(1) + 1e-12))\n    lo, hi = np.percentile(db, 10), np.percentile(db, 95); act = db > lo + 0.5 * (hi - lo)\n    c = np.minimum(((np.arange(k) + 0.5) * 0.05 // 2).astype(int), n - 1)\n    return [round(float(act[c == j].mean()), 3) if (c == j).any() else 0.0 for j in range(n)]\n\n\ndef item(i, r):\n    src, a, dur, w = prep_clip(r, MAXD)\n    media = prepare_video(src); stem, texts = parse_question(r[\'question\']); task = r[\'task_type_abbr\']\n    rec = dict(dur=round(dur, 2), win=[round(w[0], 2), round(w[1], 2)], speech_first=speech_first(w, dur))\n    P = profiles(media, prompt_of(stem, texts, tuple(range(len(texts)))))\n    P0 = profiles(media, NEUTRAL)\n    rel = {\'a\': P[\'a\'], \'a_pmi\': pmi(P[\'a\'], P0[\'a\']), \'av_pmi\': pmi(P[\'av\'], P0[\'av\']), \'v_pmi\': pmi(P[\'v\'], P0[\'v\'])}\n    nc = len(P[\'a\']); act = np.array(chunk_activity(a, nc)) > 0.3; rec[\'speech_chunks\'] = act.astype(int).tolist()\n    for k_, p in list(rel.items()) + [(\'v\', P[\'v\']), (\'av\', P[\'av\'])]:\n        s0, s1 = top_window(p); wv = np.zeros(nc, bool); wv[s0:s1] = True\n        rec[f\'rel_{k_}\'] = np.round(p, 4).tolist(); rec[f\'rel_{k_}_iou\'] = round(float((wv & act).sum() / max(1, (wv | act).sum())), 3)\n    rec[\'neutral_a\'] = np.round(P0[\'a\'], 4).tolist(); rec[\'neutral_v\'] = np.round(P0[\'v\'], 4).tolist()\n\n    def mk(fn):\n        def f(inp):\n            lay = Lay(inp); return forward(inp, lay, fn(lay))[0]\n        return f\n    def sg(lay):   # both modalities re-weighted in layers 13-20; the upper routed layers additionally read with the sound off\n        both = bias_mask(lay, rel[\'av_pmi\'], 4.0, (\'v\', \'a\')); off = bias_mask(lay, rel[\'av_pmi\'], 4.0, (\'v\',), block_audio=True)\n        return {li: (both if li < 21 else off) for li in ROUTE}\n    rec[\'base\'] = score(media, stem, texts)\n    rec[\'oracle_route\'] = score(media, stem, texts, fwd=mk(lambda lay: window_mask(lay, w[0], w[1], True)))\n    rec[\'acar_a_l4_ab\'] = score(media, stem, texts, fwd=mk(lambda lay: bias_mask(lay, rel[\'a\'], 4.0, (\'v\',), True)))\n    rec[\'acar_a_pmi_l4_ab\'] = score(media, stem, texts, fwd=mk(lambda lay: bias_mask(lay, rel[\'a_pmi\'], 4.0, (\'v\',), True)))\n    rec[\'qtar_av_pmi_l1\'] = score(media, stem, texts, fwd=mk(lambda lay: bias_mask(lay, rel[\'av_pmi\'], 1.0, (\'v\', \'a\'))))\n    rec[\'qtar_av_pmi_l4\'] = score(media, stem, texts, fwd=mk(lambda lay: bias_mask(lay, rel[\'av_pmi\'], 4.0, (\'v\', \'a\'))))\n    rec[\'qtar_av_pmi_l4_sg\'] = score(media, stem, texts, fwd=mk(sg))\n    if task in (\'TA\', \'TF\'):\n        s0, s1 = top_window(rel[\'a_pmi\']); T0, T1 = 2.0 * s0, min(dur, 2.0 * s1); look = {}\n        for name, t0, t1 in ((\'before\', 0.0, T0), (\'during\', T0, T1), (\'after\', T1, dur)):\n            if t1 - t0 < 1.0: continue\n            s = score(media, LOOK_Q, [\'Yes\', \'No\'], fwd=mk(lambda lay, t0=t0, t1=t1: window_mask(lay, t0, t1, False)))\n            look[name] = round(s[0] - s[1], 3)\n        rec[\'look\'] = look\n    return rec\n\n\nitems = sample_items((\'TA\', \'TF\', \'CMTB\'), PER_LABEL, seed=SEED)\nC = (\'base\', \'oracle_route\', \'acar_a_l4_ab\', \'acar_a_pmi_l4_ab\', \'qtar_av_pmi_l1\', \'qtar_av_pmi_l4\', \'qtar_av_pmi_l4_sg\')\nR = run_loop(items, \'j8_acar.jsonl\', item, show=C)\nOK = [r for r in R if \'error\' not in r]\nRESULT = {t: {c: acc(OK, c, t)[0] for c in C} | {\'n\': sum(r[\'task\'] == t for r in OK)} for t in (\'TA\', \'TF\', \'CMTB\')}\nRESULT[\'ALL\'] = {c: acc(OK, c)[0] for c in C}\nRESULT[\'relevance_window_iou_with_speech_median\'] = {k: round(float(np.median([r[f\'rel_{k}_iou\'] for r in OK])), 3)\n                                                    for k in (\'a\', \'a_pmi\', \'v\', \'v_pmi\', \'av\', \'av_pmi\')} if OK else None\n\n\ndef look_rule(r):\n    lk = r.get(\'look\') or {}\n    if not lk: return None\n    best = max(lk, key=lk.get)\n    if r[\'task\'] == \'TA\': return 0 if best == \'during\' else 1\n    return {\'before\': 0, \'during\': 1, \'after\': 2}[best]\nL = [r for r in OK if r.get(\'look\')]\nRESULT[\'look_rule_acc\'] = {t: round(100 * float(np.mean([look_rule(r) == r[\'gold\'] for r in L if r[\'task\'] == t])), 1) for t in (\'TA\', \'TF\') if any(r[\'task\'] == t for r in L)}\nprint(RESULT, flush=True)\n'}
for _n, _s in SRC.items(): (ROOT/_n).write_text(_s, encoding='utf-8')
print('Đã ghi:', list(SRC), '| params', PARAMS)


In [ ]:
# CELL 3 — chạy thử 2 mẫu (dừng nếu lỗi)
import subprocess, json as _json
def run_job(out_dir, params):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(PYTHON), str(ROOT/'nb_runner.py'), str(ROOT), str(out_dir), str(ROOT/JOB), _json.dumps(params)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        low = line.lower()
        if ('warn' in low or 'loading checkpoint' in low or 'fetching' in low or 'it/s]' in low) and 'traceback' not in low: continue
        print(line, end='', flush=True)
    return p.wait()
SMOKE = OUT/'smoke'
code = run_job(SMOKE, {**PARAMS, 'LIMIT': 4, 'HOURS': 1.0})
rows = [_json.loads(l) for f in SMOKE.glob('*.jsonl') for l in f.read_text().splitlines() if l.strip()]
good = [r for r in rows if 'error' not in r]
real_err = [r for r in rows if 'error' in r and 'long' not in r['error'] and 'no room' not in r['error']]   # 'long' = clip over the length limit, not a bug
print('\nSMOKE: exit', code, '| items', len(rows), '| ok', len(good), '| real errors', len(real_err))
for r in rows:
    if 'error' in r: print('  error:', r['error'][:200])
if code != 0 or real_err:
    raise RuntimeError('Smoke test thất bại: dừng để không phí GPU. Gửi log của cell này cho Claude.')
print('Smoke test OK -> chạy job đầy đủ.' if good else 'Cả 4 clip thử đều quá dài (không phải lỗi) -> vẫn chạy job đầy đủ.')


In [ ]:
# CELL 4 — chạy job đầy đủ (tự dừng trước hạn)
import subprocess, json as _json
def run_job(out_dir, params):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(PYTHON), str(ROOT/'nb_runner.py'), str(ROOT), str(out_dir), str(ROOT/JOB), _json.dumps(params)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        low = line.lower()
        if ('warn' in low or 'loading checkpoint' in low or 'fetching' in low or 'it/s]' in low) and 'traceback' not in low: continue
        print(line, end='', flush=True)
    return p.wait()
code = run_job(OUT/'full', PARAMS)
print('exit code', code)


In [ ]:
# CELL 5 — tóm tắt
for f in (OUT/"full").glob("*.jsonl"):
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    print(f.name, "| items", len(rows), "| errors", sum("error" in r for r in rows))


In [ ]:
# CELL 6 — nén kết quả (jsonl + npz + log; không kèm video)
ZIP = Path(f'/kaggle/working/{ZIPNAME}.zip')
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (OUT/'full').rglob('*'):
        if f.is_file() and f.suffix in ('.jsonl', '.json', '.npz', '.log', '.pt') and 'clips' not in f.parts:
            z.write(f, str(f.relative_to(OUT/'full')))
    for f in (OUT/'smoke').glob('*.jsonl'): z.write(f, 'smoke/' + f.name)
print(ZIP, round(ZIP.stat().st_size / 1e6, 1), 'MB')
display(FileLink(str(ZIP)))
